# Practical Assignment: Multi-Layer Perceptrons (50 Points)

## Grading Distribution & Assessment Criteria

| Section | Target Task | Evaluation Metric | Points |
| :--- | :--- | :--- | :---: |
| **Part 1.1** | `FullyConnectedFromScratch.forward` & `compute_loss` | Step-by-step numerical assertion match | 4 |
| **Part 1.2** | `FullyConnectedFromScratch.backward` (Local Gradients & Momentum) | Analytical vs. Numerical Gradient Check ($< 10^{-5}$) | 8 |
| **Part 1.3** | Mini-batch Vectorization & Shape Invariance | Dynamic batch-size dimension tests | 3 |
| **Part 2.1** | Symmetry Breaking Analysis (Zero vs. Random Init on XOR) | Convergence threshold & classification correctness | 5 |
| **Part 2.2** | `BoldDriverOptimizer.adapt` Implementation | Directional learning rate adaptation unit test | 5 |
| **Part 3.1** | `RegularizedMLP` Architecture & Operational Modes | Layer registration and Dropout stochasticity checks | 4 |
| **Part 3.2** | `EarlyStopping` Monitor Logic | Validation loss stagnation and checkpoint rollback | 4 |
| **Part 3.3** | Comparative Regularization Experiments on Fashion-MNIST | Generalization gap reduction validation | 7 |
| **Part 4.1** | Non-Linear Decision Boundary Pipeline (Circles Dataset) | Complete enclosed decision region formation | 5 |
| **Part 4.2** | Geometric Capacity Analysis & Interior Cluster Isolation | Convex polytope classification assertion | 5 |
| **Total** | | | **50** |

### Submission Rules:
- Complete all sections marked with `### YOUR CODE HERE`.
- Do not modify or delete assertion and testing cells.
- Your final submitted notebook must run sequentially from top to bottom (`Restart Kernel and Run All Cells`) without errors.

# Multi-Layer Perceptrons: From Foundations to Regularization

## Objectives
1. Implement a 2-layer Perceptron from scratch using purely NumPy, verifying analytical backpropagation against theoretical calculations.
2. Empirically investigate weight symmetry, activation saturation, and momentum dynamics.
3. Construct a standard deep learning pipeline in PyTorch to evaluate regularization techniques (Weight Decay, Dropout, Early Stopping).
4. Visualize geometric decision boundaries to understand capacity and non-linear separability.

## Instructions
- Do not import external deep learning libraries (e.g., PyTorch, TensorFlow) in Parts 1 and 2. Use pure NumPy.
- PyTorch is permitted and required only in Part 3 and Part 4.
- All evaluation cells contain deterministic assertions; ensure all test cells pass.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

# PyTorch imports for Parts 3 & 4
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

# Reproducibility configuration
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration (Compatible with Colab, Kaggle, and Local)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Execution environment configured. Active device: {device}")

## Part 1: Analytical Forward and Backward Propagation (Pure NumPy)

In this section, you will implement a 2-layer feedforward network with:
- Input dimension: $d_{in}$
- Hidden layer dimension: $d_h$ (Sigmoid activation)
- Output dimension: $d_{out}$ (Sigmoid activation)
- Objective: Mean Squared Error (MSE) loss:
$$E = \frac{1}{2} \sum_{k} (d_k - \hat{y}_k)^2$$

### Equations
1. Hidden net input and activations:
   $$v_h = W_h x + b_h, \quad h = \sigma(v_h) = \frac{1}{1 + e^{-v_h}}$$
2. Output net input and predictions:
   $$v_{out} = W_{out} h + b_{out}, \quad \hat{y} = \sigma(v_{out})$$
3. Local gradients (Delta Rule):
   $$\delta_{out} = \hat{y} \odot (1 - \hat{y}) \odot (d - \hat{y})$$
   $$\delta_{h} = h \odot (1 - h) \odot (W_{out}^T \delta_{out})$$
4. Weight and bias updates with learning rate $\eta$ and momentum $\alpha$:
   $$\Delta W(n) = \alpha \Delta W(n-1) + \eta \, \delta \, \text{input}^T$$
   $$W(n) = W(n-1) + \Delta W(n)$$

In [ ]:
class FullyConnectedFromScratch:
    def __init__(self, in_features, hidden_features, out_features, lr=0.5, momentum=0.0):
        self.in_features = in_features
        self.hidden_features = hidden_features
        self.out_features = out_features
        self.lr = lr
        self.momentum = momentum
        
        # Initialize weights and biases
        self.init_weights()
        
        # Momentum velocity buffers
        self.v_W_h = np.zeros_like(self.W_h)
        self.v_b_h = np.zeros_like(self.b_h)
        self.v_W_out = np.zeros_like(self.W_out)
        self.v_b_out = np.zeros_like(self.b_out)
        
    def init_weights(self):
        # Small random initialization
        self.W_h = np.random.randn(self.hidden_features, self.in_features) * 0.1
        self.b_h = np.zeros((self.hidden_features, 1))
        self.W_out = np.random.randn(self.out_features, self.hidden_features) * 0.1
        self.b_out = np.zeros((self.out_features, 1))
        
    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))
    
    @staticmethod
    def sigmoid_derivative(a):
        # Derivative in terms of activated value a = sigmoid(z)
        return a * (1.0 - a)
    
    def forward(self, x):
        """
        Computes the forward pass of the 2-layer network.
        x: Input array of shape (in_features, batch_size)
        Returns:
            y_hat: Network output activations of shape (out_features, batch_size)
        """
        self.x = x
        # ### YOUR CODE HERE ###
        # 1. Compute hidden net input v_h and activation h using self.sigmoid
        # 2. Compute output net input v_out and activation y_hat using self.sigmoid
        raise NotImplementedError("Implement forward pass.")
        # ######################
        return self.y_hat
    
    def compute_loss(self, target):
        return 0.5 * np.sum((target - self.y_hat) ** 2)
    
    def backward(self, target):
        """
        Computes local gradients and parameter updates according to the Generalized Delta Rule.
        target: Ground truth array of shape (out_features, batch_size)
        """
        batch_size = self.x.shape[1] if self.x.ndim > 1 else 1
        # ### YOUR CODE HERE ###
        # 1. Compute delta_out using self.sigmoid_derivative
        # 2. Compute delta_h using self.sigmoid_derivative and self.W_out
        # 3. Compute gradients for W_out, b_out, W_h, b_h normalized by batch_size
        # 4. Update velocity buffers (momentum) and apply updates to weights and biases
        raise NotImplementedError("Implement backward pass.")
        # ######################

In [ ]:
# Verification using the exact numerical parameters from ML_HW2_MLP_1405 Question 4
model_verify = FullyConnectedFromScratch(in_features=2, hidden_features=2, out_features=1, lr=0.5, momentum=0.0)

# Inject predetermined parameters from Question 4
model_verify.W_h = np.array([[0.5, -0.2], 
                             [-0.4, 0.3]])
model_verify.b_h = np.array([[0.1], 
                             [-0.2]])
model_verify.W_out = np.array([[0.6, -0.8]])
model_verify.b_out = np.array([[0.2]])

# Input x = [1, 0]^T and target d = [1]^T
x_sample = np.array([[1.0], [0.0]])
target_sample = np.array([[1.0]])

# Forward pass
y_pred = model_verify.forward(x_sample)
loss = model_verify.compute_loss(target_sample)

# Backward pass
grads = model_verify.backward(target_sample)

# Assertions matching theoretical step-by-step results with proper floating-point tolerance
np.testing.assert_allclose(model_verify.v_h, np.array([[0.6], [-0.6]]), atol=1e-3, err_msg="v_h calculation mismatch")
np.testing.assert_allclose(model_verify.h, np.array([[0.646], [0.354]]), atol=1e-3, err_msg="Hidden activations mismatch")
np.testing.assert_allclose(model_verify.v_out, np.array([[0.305]]), atol=2e-3, err_msg="v_out calculation mismatch")
np.testing.assert_allclose(y_pred, np.array([[0.576]]), atol=2e-3, err_msg="Prediction y_hat mismatch")
np.testing.assert_allclose(loss, 0.090, atol=2e-3, err_msg="Loss value mismatch")

np.testing.assert_allclose(grads['delta_out'], np.array([[0.104]]), atol=2e-3, err_msg="delta_out mismatch")
np.testing.assert_allclose(grads['delta_h'], np.array([[0.014], [-0.019]]), atol=2e-3, err_msg="delta_h mismatch")
np.testing.assert_allclose(model_verify.W_out[0, 0], 0.634, atol=2e-3, err_msg="Updated W_out_1 mismatch")
np.testing.assert_allclose(model_verify.W_h[0, 0], 0.507, atol=2e-3, err_msg="Updated W_h_11 mismatch")
np.testing.assert_allclose(grads['delta_W_h'][0, 1], 0.0, atol=1e-7, err_msg="Delta W_h_12 should be zero since x2=0")

print("Part 1 verification passed: Forward, backward, and parameter updates match theoretical solutions.")

### 1.2 Gradient Verification via Finite Differences

To ensure that your analytical derivative derivations in `backward()` are mathematically sound, we employ numerical gradient checking. 
Using the central difference formula:
$$\frac{\partial E}{\partial w_{ij}} \approx \frac{E(w_{ij} + \epsilon) - E(w_{ij} - \epsilon)}{2\epsilon}$$
where $\epsilon = 10^{-5}$. 

The relative error metric is defined as:
$$\text{Relative Error} = \frac{\|\nabla_{analytical} - \nabla_{numerical}\|_2}{\|\nabla_{analytical}\|_2 + \|\nabla_{numerical}\|_2 + 10^{-12}}$$

A correct implementation must achieve a relative error below $10^{-5}$.

In [ ]:
def numerical_gradient_check(model, x, target, epsilon=1e-5):
    """
    Validates analytical gradients via finite difference approximation.
    Computes relative error: ||grad_num - grad_ana|| / (||grad_num|| + ||grad_ana||)
    """
    batch_size = x.shape[1] if x.ndim > 1 else 1
    
    # 1. Forward pass at original weights
    model.forward(x)
    
    # 2. Compute analytical cost gradients: dE/dW = - (delta @ input.T) / batch_size
    # Slide 13-14: delta_j = - dE/dv_j  ==>  dE/dW_ji = - delta_j * y_i
    delta_out = model.sigmoid_derivative(model.y_hat) * (target - model.y_hat)
    delta_h = model.sigmoid_derivative(model.h) * np.dot(model.W_out.T, delta_out)
    
    analytical_grad_W_out = - np.dot(delta_out, model.h.T) / batch_size
    analytical_grad_W_h = - np.dot(delta_h, model.x.T) / batch_size
    
    # 3. Compute numerical gradients via central differences without mutating model state
    numerical_grad_W_out = np.zeros_like(model.W_out)
    for i in range(model.W_out.shape[0]):
        for j in range(model.W_out.shape[1]):
            orig_val = model.W_out[i, j]
            
            model.W_out[i, j] = orig_val + epsilon
            model.forward(x)
            loss_plus = model.compute_loss(target) / batch_size
            
            model.W_out[i, j] = orig_val - epsilon
            model.forward(x)
            loss_minus = model.compute_loss(target) / batch_size
            
            model.W_out[i, j] = orig_val
            numerical_grad_W_out[i, j] = (loss_plus - loss_minus) / (2 * epsilon)
            
    numerical_grad_W_h = np.zeros_like(model.W_h)
    for i in range(model.W_h.shape[0]):
        for j in range(model.W_h.shape[1]):
            orig_val = model.W_h[i, j]
            
            model.W_h[i, j] = orig_val + epsilon
            model.forward(x)
            loss_plus = model.compute_loss(target) / batch_size
            
            model.W_h[i, j] = orig_val - epsilon
            model.forward(x)
            loss_minus = model.compute_loss(target) / batch_size
            
            model.W_h[i, j] = orig_val
            numerical_grad_W_h[i, j] = (loss_plus - loss_minus) / (2 * epsilon)
            
    # Reset model predictions to original state
    model.forward(x)
    
    def rel_error(ana, num):
        return np.linalg.norm(ana - num) / (np.linalg.norm(ana) + np.linalg.norm(num) + 1e-12)
    
    err_W_out = rel_error(analytical_grad_W_out, numerical_grad_W_out)
    err_W_h = rel_error(analytical_grad_W_h, numerical_grad_W_h)
    
    return err_W_out, err_W_h

# Execute gradient check on batch data
np.random.seed(0)
test_net = FullyConnectedFromScratch(in_features=3, hidden_features=4, out_features=2, lr=0.1, momentum=0.0)
X_test = np.random.randn(3, 5)
Y_test = np.random.uniform(0.1, 0.9, (2, 5))

err_out, err_h = numerical_gradient_check(test_net, X_test, Y_test)

assert err_out < 1e-5, f"Gradient check failed for W_out. Relative error: {err_out:.2e}"
assert err_h < 1e-5, f"Gradient check failed for W_h. Relative error: {err_h:.2e}"
print(f"Gradient check passed: W_out error = {err_out:.2e}, W_h error = {err_h:.2e}")

### 1.3 Batch Processing & Vectorization Invariance

Deep learning workloads process mini-batches of size $B$ concurrently. Your implementation must not rely on hardcoded single-sample dimensions. 
The test below evaluates forward and backward passes across varying batch sizes $B \in \{1, 4, 16, 64\}$ to verify that:
1. Output dimensions match $(d_{out}, B)$.
2. Parameter shapes ($W_h, b_h, W_{out}, b_{out}$) remain invariant regardless of batch size.

In [ ]:
# Batch processing and dimension sanity check
batch_sizes = [1, 4, 16, 64]
in_f, hid_f, out_f = 5, 8, 3

model_batch = FullyConnectedFromScratch(in_features=in_f, hidden_features=hid_f, out_features=out_f)

for b_size in batch_sizes:
    x_b = np.random.randn(in_f, b_size)
    y_b = np.random.uniform(0.1, 0.9, (out_f, b_size))
    
    out_b = model_batch.forward(x_b)
    assert out_b.shape == (out_f, b_size), f"Forward output shape mismatch for batch size {b_size}. Expected {(out_f, b_size)}, got {out_b.shape}"
    
    grads = model_batch.backward(y_b)
    assert model_batch.W_h.shape == (hid_f, in_f), "W_h shape corrupted during backward step"
    assert model_batch.b_h.shape == (hid_f, 1), "b_h shape corrupted during backward step"
    assert model_batch.W_out.shape == (out_f, hid_f), "W_out shape corrupted during backward step"
    assert model_batch.b_out.shape == (out_f, 1), "b_out shape corrupted during backward step"

print("Batch invariance test passed: Arbitrary batch dimensions processed correctly.")

## Part 2: Training Dynamics, Symmetry Breaking, and Saturation

In this section, you will analyze:
1. **Symmetry Breaking:** Comparing zero weight initialization versus random initialization.
2. **Sigmoid Saturation & Vanishing Gradients:** Monitoring activation derivatives as net inputs become large.
3. **Adaptive Learning Rates:** Implementing the Bold Driver heuristic to adjust $\eta$ dynamically based on error progression.

In [ ]:
# Generate XOR dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]).T
y_xor = np.array([[0, 1, 1, 0]])

def train_xor_experiment(init_type='random', epochs=10000, lr=0.8, momentum=0.9):
    net = FullyConnectedFromScratch(in_features=2, hidden_features=2, out_features=1, lr=lr, momentum=momentum)
    
    if init_type == 'zero':
        net.W_h = np.zeros((2, 2))
        net.b_h = np.zeros((2, 1))
        net.W_out = np.zeros((1, 2))
        net.b_out = np.zeros((1, 1))
        net.v_W_h.fill(0)
        net.v_b_h.fill(0)
        net.v_W_out.fill(0)
        net.v_b_out.fill(0)
    else:
        # Proper initialization range [-1.0, 1.0] according to Slide 30
        np.random.seed(42)
        net.W_h = np.random.uniform(-1.0, 1.0, (2, 2))
        net.b_h = np.zeros((2, 1))
        net.W_out = np.random.uniform(-1.0, 1.0, (1, 2))
        net.b_out = np.zeros((1, 1))
        
    losses = []
    for epoch in range(epochs):
        pred = net.forward(X_xor)
        loss = net.compute_loss(y_xor) / 4.0
        net.backward(y_xor)
        losses.append(loss)
        
    return net, losses

net_zero, losses_zero = train_xor_experiment('zero')
net_rand, losses_rand = train_xor_experiment('random')

# Assertions verifying symmetry failure and random initialization success
assert np.all(net_zero.W_h[0, :] == net_zero.W_h[1, :]), "Zero-initialized hidden neurons must remain identical due to symmetry."
assert losses_zero[-1] > 0.1, "Zero-initialized network must fail to separate XOR due to symmetry."

# Check convergence and complete separation of the XOR problem
predictions_rand = (net_rand.forward(X_xor) > 0.5).astype(int)
assert np.array_equal(predictions_rand, y_xor), "Properly initialized network must correctly classify all 4 XOR states."
assert losses_rand[-1] < 0.02, f"Training loss should drop below 0.02, but reached {losses_rand[-1]:.4f}."

# Plot convergence comparison
plt.figure(figsize=(8, 4))
plt.plot(losses_zero, label='Zero Initialization (Symmetric)', color='crimson')
plt.plot(losses_rand, label='Random Initialization (Symmetry Broken, Momentum=0.9)', color='forestgreen')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error')
plt.title('Impact of Symmetry Breaking and Momentum on XOR Convergence')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f"Final Loss (Zero Init): {losses_zero[-1]:.4f}")
print(f"Final Loss (Random Init): {losses_rand[-1]:.4f}")
print("Predictions:", net_rand.forward(X_xor).ravel())

> ### Expected Output Reference: XOR Convergence & Symmetry Breaking
> A correct implementation should produce loss trajectories consistent with the reference plot below:
> - **Zero Initialization:** Stuck indefinitely at an error plateau of ~0.1250 due to neuron symmetry.
> - **Random Initialization (with Momentum = 0.9):** Successfully escapes the saddle point and converges to an MSE below 0.01.
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW2-MLP/assets/xor_convergence_ref.png" width="650"/>
> </p>

In [ ]:
class BoldDriverOptimizer:
    """
    Implements Bold Driver heuristic:
    - If current loss < previous loss: increase lr by factor rho_inc (e.g. 1.05)
    - If current loss > previous loss: decrease lr significantly by factor rho_dec (e.g. 0.5) and reject update
    """
    def __init__(self, initial_lr=0.1, rho_inc=1.05, rho_dec=0.5):
        self.lr = initial_lr
        self.rho_inc = rho_inc
        self.rho_dec = rho_dec
        self.prev_loss = None
        
    def adapt(self, current_loss):
        """
        Applies Bold Driver heuristic:
        - If loss decreases: increase lr by factor rho_inc, accept step.
        - If loss increases: decrease lr by factor rho_dec, reject step.
        Returns:
            (updated_lr, reject_step_flag)
        """
        # ### YOUR CODE HERE ###
        raise NotImplementedError("Implement Bold Driver adaptation.")
        # ######################

# Verification unit test for Bold Driver
bd = BoldDriverOptimizer(initial_lr=0.2, rho_inc=1.05, rho_dec=0.5)
lr_step1, reject1 = bd.adapt(0.50)
lr_step2, reject2 = bd.adapt(0.45) # loss decreased -> lr should increase
assert np.isclose(lr_step2, 0.2 * 1.05), "Bold driver should increase lr when loss decreases"
assert not reject2, "Step must not be rejected on decrease"

lr_step3, reject3 = bd.adapt(0.48) # loss increased -> lr should drop to 50%
assert np.isclose(lr_step3, (0.2 * 1.05) * 0.5), "Bold driver should halve lr when loss increases"
assert reject3, "Step must be rejected on error increase"
print("Part 2 verification passed: Bold Driver heuristic validated successfully.")

## Part 3: PyTorch Modular Pipelines & Overfitting Remedies

In this section, we transition to PyTorch to systematically evaluate three regularization mechanisms on Fashion-MNIST:
1. **Unregularized Baseline (Large Network):** 2 hidden layers (512, 256 units), susceptible to overfitting when trained with high capacity.
2. **Weight Decay ($L_2$ Regularization):** Adding parameter penalty $\lambda \sum w^2$.
3. **Dropout:** Zeroing hidden neuron activations randomly during training with drop probability $p$.
4. **Early Stopping:** Halting training upon validation error stagnation to preserve generalization.

In [ ]:
def load_fashion_mnist(batch_size=128, subset_size=5000):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    
    # Download dataset locally or within cloud notebook
    train_full = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    
    # Subsample training data to simulate data scarcity and induce controlled overfitting
    train_subset, val_subset = torch.utils.data.random_split(
        torch.utils.data.Subset(train_full, range(subset_size)),
        [int(0.8 * subset_size), int(0.2 * subset_size)],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = load_fashion_mnist(batch_size=128, subset_size=4000)
print("Fashion-MNIST data loaders initialized successfully.")

In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, in_features=784, hidden1=256, hidden2=128, num_classes=10, dropout_rate=0.0):
        super(RegularizedMLP, self).__init__()
        self.flatten = nn.Flatten()
        
        self.fc1 = nn.Linear(in_features, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_rate)
        
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=dropout_rate)
        
        self.out = nn.Linear(hidden2, num_classes)
        
    def forward(self, x):
        x = self.flatten(x)
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        logits = self.out(x)
        return logits

### 3.2 Model Verification: Architecture and Operational Modes

Before initiating training, models must be validated against two critical failure modes:
1. **Layer Topology & Parameter Count:** Verifying that linear layers and activation sequences correspond strictly to the specified architecture.
2. **Train vs. Evaluation Invariance:** Regularization layers such as `nn.Dropout` must exhibit stochasticity during training (`model.train()`), but must act deterministically and preserve activation scales during evaluation (`model.eval()`).

In [ ]:
# Architecture and forward shape verification for PyTorch MLP
sample_input = torch.randn(8, 1, 28, 28)
mlp_test = RegularizedMLP(in_features=784, hidden1=256, hidden2=128, num_classes=10, dropout_rate=0.2)

# Ensure parameters are correctly registered
param_count = sum(p.numel() for p in mlp_test.parameters() if p.requires_grad)
expected_param_count = (784 * 256 + 256) + (256 * 128 + 128) + (128 * 10 + 10)
assert param_count == expected_param_count, f"Parameter count mismatch. Expected {expected_param_count}, got {param_count}"

# Test forward output
output_tensor = mlp_test(sample_input)
assert output_tensor.shape == (8, 10), f"Output tensor shape mismatch. Expected (8, 10), got {output_tensor.shape}"
print(f"Architecture sanity test passed: Total trainable parameters = {param_count}")

In [ ]:
# Dropout operational behavior test
set_seed(42)
dropout_test_model = RegularizedMLP(in_features=10, hidden1=50, hidden2=50, num_classes=2, dropout_rate=0.5)
dummy_x = torch.randn(10, 10)

# Mode: train -> consecutive passes should yield different outputs due to random masks
dropout_test_model.train()
out_train_1 = dropout_test_model(dummy_x)
out_train_2 = dropout_test_model(dummy_x)
assert not torch.allclose(out_train_1, out_train_2), "Dropout must introduce stochasticity during training mode"

# Mode: eval -> consecutive passes must be strictly deterministic
dropout_test_model.eval()
with torch.no_grad():
    out_eval_1 = dropout_test_model(dummy_x)
    out_eval_2 = dropout_test_model(dummy_x)
assert torch.allclose(out_eval_1, out_eval_2, atol=1e-7), "Model must behave deterministically in evaluation mode (Dropout inactive)"

print("Dropout mode test passed: Determinism in eval mode and stochasticity in train mode verified.")

In [ ]:
class EarlyStopping:
    """
    Monitors validation loss and stops training if no improvement is observed for 'patience' epochs.
    """
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False
        self.best_state = None
        
    def step(self, val_loss, model):
        """
        Monitors validation loss. If no improvement for 'patience' epochs, sets self.early_stop to True.
        Saves the best model state dictionary.
        """
        # ### YOUR CODE HERE ###
        raise NotImplementedError("Implement early stopping logic.")
        # ######################

# Sanity assertion for EarlyStopping
dummy_model = nn.Linear(2, 1)
stopper = EarlyStopping(patience=3)
stopper.step(1.0, dummy_model)
stopper.step(0.9, dummy_model)
stopper.step(0.91, dummy_model)
stopper.step(0.92, dummy_model)
stopper.step(0.93, dummy_model)
assert stopper.early_stop is True, "Early stopping should trigger when validation loss does not improve for 'patience' epochs"
print("EarlyStopping utility verified.")

In [ ]:
from tqdm.auto import tqdm

def train_and_evaluate(model, train_loader, val_loader, epochs=30, lr=0.01, 
                       weight_decay=0.0, use_early_stopping=False, patience=5, desc="Training"):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    stopper = EarlyStopping(patience=patience) if use_early_stopping else None
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    pbar = tqdm(range(epochs), desc=desc, leave=True)
    for epoch in pbar:
        # Training Phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            
        train_loss = running_loss / total
        train_acc = correct / total
        
        # Validation Phase
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_running_loss += loss.item() * X_batch.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
                
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        pbar.set_postfix({
            'TrLoss': f"{train_loss:.3f}", 
            'ValLoss': f"{val_loss:.3f}", 
            'ValAcc': f"{val_acc:.3f}"
        })
        
        if stopper:
            stopper.step(val_loss, model)
            if stopper.early_stop:
                model.load_state_dict(stopper.best_state)
                break
                
    return history

In [ ]:
# Experiment 1: Overfitted Baseline (No Regularization)
set_seed(42)
baseline_net = RegularizedMLP(dropout_rate=0.0)
hist_baseline = train_and_evaluate(baseline_net, train_loader, val_loader, epochs=40, lr=0.05, weight_decay=0.0, desc="Exp 1: Baseline")

# Experiment 2: Weight Decay (L2 = 0.01)
set_seed(42)
l2_net = RegularizedMLP(dropout_rate=0.0)
hist_l2 = train_and_evaluate(l2_net, train_loader, val_loader, epochs=40, lr=0.05, weight_decay=0.01, desc="Exp 2: L2 Decay")

# Experiment 3: Dropout (p = 0.3)
set_seed(42)
dropout_net = RegularizedMLP(dropout_rate=0.3)
hist_dropout = train_and_evaluate(dropout_net, train_loader, val_loader, epochs=40, lr=0.05, weight_decay=0.0, desc="Exp 3: Dropout")

# Experiment 4: Early Stopping (Patience = 4)
set_seed(42)
es_net = RegularizedMLP(dropout_rate=0.0)
hist_es = train_and_evaluate(es_net, train_loader, val_loader, epochs=40, lr=0.05, weight_decay=0.0, use_early_stopping=True, patience=4, desc="Exp 4: Early Stopping")

print("All comparative training runs completed.")

In [ ]:
plt.figure(figsize=(14, 5))

# Plot Validation Loss Curves
plt.subplot(1, 2, 1)
plt.plot(hist_baseline['val_loss'], label='Baseline (No Reg)', color='crimson')
plt.plot(hist_l2['val_loss'], label='Weight Decay (L2=0.01)', color='navy')
plt.plot(hist_dropout['val_loss'], label='Dropout (p=0.3)', color='darkorange')
plt.plot(hist_es['val_loss'], label='Early Stopping', color='forestgreen', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss Progression Across Regularization Schemes')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot Generalization Gap: Train Acc - Val Acc
plt.subplot(1, 2, 2)
gap_baseline = np.array(hist_baseline['train_acc']) - np.array(hist_baseline['val_acc'])
gap_l2 = np.array(hist_l2['train_acc']) - np.array(hist_l2['val_acc'])
gap_dropout = np.array(hist_dropout['train_acc']) - np.array(hist_dropout['val_acc'])

plt.plot(gap_baseline, label='Baseline Gap', color='crimson')
plt.plot(gap_l2, label='Weight Decay Gap', color='navy')
plt.plot(gap_dropout, label='Dropout Gap', color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('Generalization Gap (Train Acc - Val Acc)')
plt.title('Overfitting Analysis (Generalization Gap)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Quantitative assertion confirming regularization reduces generalization gap
final_gap_base = gap_baseline[-1]
final_gap_l2 = gap_l2[-1]
assert final_gap_l2 < final_gap_base, "Weight decay must quantitatively reduce the overfitting gap compared to baseline."
print(f"Validation verified: Baseline Gap = {final_gap_base:.3f}, L2 Regulated Gap = {final_gap_l2:.3f}")

> ### Expected Output Reference: Regularization & Generalization Gap
> Compare your generated loss curves and generalization gaps with the target behaviors below:
> - **Baseline:** Validation loss diverges after epoch ~8, and the generalization gap steadily expands above 0.15 (clear overfitting).
> - **Weight Decay & Dropout:** Validation loss remains bounded, and the final generalization gap is constrained significantly below the baseline.
> - **Early Stopping:** Training terminates early as soon as validation loss fails to decrease for 4 consecutive epochs.
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW2-MLP/assets/regularization_comparison_ref.png" width="850"/>
> </p>

## Part 4: Decision Boundary Geometry & Network Capacity

Reflecting Slides 3-4 and Theoretical Homework Question 1:
- A single perceptron can only produce a linear separating hyperplane.
- A 1-hidden-layer network can isolate a single convex region.
- A 2-hidden-layer network can model arbitrary non-convex geometries and unions of convex regions.

Below, we train MLPs with varying hidden layer counts and units on a non-linear synthetic dataset (nested circles) to visualize the empirical formation of decision regions.

In [ ]:
from sklearn.datasets import make_circles

# Generate concentric non-linear decision problem
X_circ, y_circ = make_circles(n_samples=600, noise=0.1, factor=0.4, random_state=42)
X_tensor = torch.tensor(X_circ, dtype=torch.float32)
y_tensor = torch.tensor(y_circ, dtype=torch.long)

circ_dataset = TensorDataset(X_tensor, y_tensor)
circ_loader = DataLoader(circ_dataset, batch_size=32, shuffle=True)

# Define models with different capacities
class SingleLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(2, 2)
    def forward(self, x): 
        return self.net(x)

# ==============================================================================
# TODO: Implement a 2-Layer MLP with non-linear activation (1 Hidden Layer)
# Architecture requirements:
# 1. Linear layer projecting from 2 inputs to 'hidden' neurons (default: 16)
# 2. Non-linear activation (e.g., nn.ReLU)
# 3. Final Linear layer projecting to 2 output classes
# ==============================================================================
class TwoLayerMLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        # ### YOUR CODE HERE ###
        raise NotImplementedError("Define the layers for TwoLayerMLP.")
        # ######################

    def forward(self, x):
        # ### YOUR CODE HERE ###
        raise NotImplementedError("Implement the forward pass for TwoLayerMLP.")
        # ######################

def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    # Send evaluation grid to the correct model execution device
    grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to(device)
    
    model.eval()
    with torch.no_grad():
        preds = model(grid_tensor).argmax(dim=1).cpu().numpy()
    Z = preds.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k', s=25)
    ax.set_title(title)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

# Model instantiation and training
m_linear = SingleLayerPerceptron()
# ### YOUR CODE HERE ### (Instantiate TwoLayerMLP with hidden=16)
m_mlp = None
# ######################

train_and_evaluate(m_linear, circ_loader, circ_loader, epochs=30, lr=0.1, desc="Linear Model")
train_and_evaluate(m_mlp, circ_loader, circ_loader, epochs=60, lr=0.1, desc="2-Layer MLP")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(m_linear, X_circ, y_circ, "Linear Model: Inability to Form Enclosed Region", axes[0])
plot_decision_boundary(m_mlp, X_circ, y_circ, "MLP (1 Hidden Layer, 16 units): Convex Boundary Enclosure", axes[1])
plt.tight_layout()
plt.show()

# Device-agnostic assertion
grid_center = torch.tensor([[0.0, 0.0]], dtype=torch.float32).to(device)
m_mlp.eval()
with torch.no_grad():
    pred_center = m_mlp(grid_center).argmax(dim=1).cpu().item()
assert pred_center == 1, "The non-linear MLP must correctly classify the interior cluster (class 1) at center (0,0)."
print("Part 4 verified: Decision boundary correctly isolates inner non-linear cluster.")

> ### Expected Output Reference: Decision Boundaries & Convex Polytope
> Verify your decision boundary plots against the reference below:
> - **Single-Layer Perceptron (Left):** Forms only a single linear boundary, proving insufficient for non-linearly separable concentric data.
> - **2-Layer MLP (Right):** Constructs an enclosed convex boundary that isolates the inner cluster (Class 1) from the outer ring (Class 0).
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW2-MLP/assets/decision_boundaries_ref.png" width="850"/>
> </p>